# NHS Referral-to-Treatment Waiting Times Analysis
## Data Understanding

**Dataset:** NHS England Consultant-led Referral to Treatment (RTT), July 2026 — Incomplete Provider data.

### Notebook purpose

Before cleaning or analysing the dataset, this notebook establishes:

- the workbook and worksheet structure;
- the correct header row;
- the unit of analysis (row grain);
- the provider and treatment-function structure;
- the meaning of the weekly waiting-time bands;
- the role of the `C_999` total row;
- basic data-quality checks;
- reconciliation of published totals against the weekly bands; and
- a first look at Gloucestershire Hospitals NHS Foundation Trust.

> **Important:** This notebook is for understanding and validating the raw data. No permanent cleaning or renaming is performed here.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)


## Locate the raw workbook

The raw NHS workbook is stored in `data/raw/`. Keeping raw source files unchanged is good analytical practice because it preserves an auditable copy of the original data.


In [2]:
file_path = "../data/raw/Incomplete-Provider-Jul26-XLSX-9M-97ylx5.xlsx"

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names


['Provider',
 'Provider with DTA',
 'IS Provider',
 'IS Provider with DTA',
 'Notes',
 'Guidance']

The workbook contains four data sheets plus documentation sheets. For this project, the main starting point is **`Provider`**, because the objective is to analyse incomplete RTT waiting lists by NHS provider and treatment function.


## Inspect the worksheet before assigning headers

Official spreadsheets often contain titles, notes, merged cells, or blank rows above the actual table. Reading a preview with `header=None` helps identify where the tabular data really begins.


In [3]:
raw_preview = pd.read_excel(
    file_path,
    sheet_name="Provider",
    header=None
)

raw_preview.head(20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Title:,Referral to Treatment (RTT) Waiting Times,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[The '+' button above reveals the full weekly breakdown],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Summary:,Monthly Referral to Treatment (RTT) waiting times for incomplete pathways.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Period:,July 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Source:,NHS England: monthly RTT data collection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Basis:,Provider,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Published:,10th Sept

Inspection shows that the true column headings begin on **Excel row 14**. In pandas, row positions are zero-based, so `header=13` is required.


## Load the Provider sheet with the correct header


In [4]:
df = pd.read_excel(
    file_path,
    sheet_name="Provider",
    header=13
)

df.shape


(3504, 120)

In [5]:
df.columns[:15].tolist()


['Unnamed: 0',
 'Region Code',
 'Provider Code',
 'Provider Name',
 'Treatment Function Code',
 'Treatment Function',
 '>0-1',
 '>1-2',
 '>2-3',
 '>3-4',
 '>4-5',
 '>5-6',
 '>6-7',
 '>7-8',
 '>8-9']

The worksheet initially contains one completely empty column (`Unnamed: 0`). We can remove only fully empty columns without otherwise changing the raw records.


In [6]:
df = df.dropna(axis=1, how="all")

df.shape


(3504, 119)

After removing the blank column, the working dataset contains **3,504 rows and 119 columns**.


## Inspect the records and important fields


In [7]:
df.head()


,Region Code,Provider Code,Provider Name,Treatment Function Code,Treatment Function,>0-1,>1-2,>2-3,>3-4,>4-5,>5-6,>6-7,>7-8,>8-9,>9-10,>10-11,>11-12,>12-13,>13-14,>14-15,>15-16,>16-17,>17-18,>18-19,>19-20,>20-21,>21-22,>22-23,>23-24,>24-25,>25-26,>26-27,>27-28,>28-29,>29-30,>30-31,>31-32,>32-33,>33-34,>34-35,>35-36,>36-37,>37-38,>38-39,>39-40,>40-41,>41-42,>42-43,>43-44,>44-45,>45-46,>46-47,>47-48,>48-49,>49-50,>50-51,>51-52,>52-53,>53-54,>54-55,>55-56,>56-57,>57-58,>58-59,>59-60,>60-61,>61-62,>62-63,>63-64,>64-65,>65-66,>66-67,>67-68,>68-69,>69-70,>70-71,>71-72,>72-73,>73-74,>74-75,>75-76,>76-77,>77-78,>78-79,>79-80,>80-81,>81-82,>82-83,>83-84,>84-85,>85-86,>86-87,>87-88,>88-89,>89-90,>90-91,>91-92,>92-93,>93-94,>94-95,>95-96,>96-97,>97-98,>98-99,>99-100,>100-101,>101-102,>102-103,>103-104,104 plus,Total number of incomplete pathways,Total within 18 weeks,% within 18 weeks,Average (median) waiting time (in weeks),92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 78 plus weeks,Total 65 plus weeks,% 52 plus weeks
0,Y56,R1H,BARTS HEALTH NHS TRUST,C_100,General Surgery Service,257,238,194,211,193,198,193,149,169,105,153,113,131,133,128,122,81,147,105,95,104,71,71,62,75,54,55,50,60,51,26,36,41,40,35,48,32,23,49,45,42,31,46,30,27,31,20,19,15,17,4,11,6,7,2,2,5,4,10,5,1,1,1,1,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4483,2915,0.650234,12.522901,39.585778,47,0,2,0.010484
1,Y56,R1H,BARTS HEALTH NHS TRUST,C_101,Urology Service,359,239,191,187,207,226,150,296,183,132,220,192,163,185,204,123,113,97,153,121,146,106,126,101,129,108,110,102,114,70,36,69,92,111,130,63,48,52,55,40,26,50,37,38,17,21,22,45,14,23,11,10,4,4,7,6,2,1,1,1,3,2,2,2,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5901,3467,0.587528,14.10049,37.517692,38,0,3,0.006440
2,Y56,R1H,BARTS HEALTH NHS TRUST,C_110,Trauma and Orthopaedic Service,396,321,329,298,300,248,272,349,301,210,320,259,225,253,260,213,205,252,204,164,170,160,139,137,159,148,176,175,190,151,52,80,130,146,138,117,101,101,87,89,95,93,76,63,45,51,50,33,24,24,16,9,9,9,8,11,13,4,6,5,3,2,1,0,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8678,5011,0.577437,14.992308,39.536629,74,0,1,0.008527
3,Y56,R1H,BARTS HEALTH NHS TRUST,C_120,Ear Nose and Throat Service,253,229,188,202,232,191,216,182,160,119,145,149,117,165,131,138,66,119,109,90,110,109,113,103,126,109,127,137,125,109,34,58,113,111,99,81,84,84,78,69,90,88,63,66,91,58,57,53,40,56,36,40,32,31,20,23,27,22,19,19,13,11,11,14,4,2,3,4,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6180,3002,0.485761,18.807339,46.870175,262,0,16,0.042395
4,Y56,R1H,BARTS HEALTH NHS TRUST,C_130,Ophthalmology Service,335,277,314,329,240,292,264,262,234,180,255,241,171,225,218,205,111,158,132,130,125,106,88,115,96,100,78,101,75,73,40,67,63,62,50,70,59,44,38,54,56,30,21,21,9,11,3,7,3,6,1,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6249,4311,0.689870,11.591286,33.791613,4,0,0,0.000640


In [8]:
key_columns = [
    "Region Code",
    "Provider Code",
    "Provider Name",
    "Treatment Function Code",
    "Treatment Function",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "Total 65 plus weeks",
    "Total 78 plus weeks",
    "% 52 plus weeks"
]

df[key_columns].head(10)


,Region Code,Provider Code,Provider Name,Treatment Function Code,Treatment Function,Total number of incomplete pathways,Total within 18 weeks,% within 18 weeks,Average (median) waiting time (in weeks),92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 65 plus weeks,Total 78 plus weeks,% 52 plus weeks
0,Y56,R1H,BARTS HEALTH NHS TRUST,C_100,General Surgery Service,4483,2915,0.650234,12.522901,39.585778,47,2,0,0.010484
1,Y56,R1H,BARTS HEALTH NHS TRUST,C_101,Urology Service,5901,3467,0.587528,14.10049,37.517692,38,3,0,0.006440
2,Y56,R1H,BARTS HEALTH NHS TRUST,C_110,Trauma and Orthopaedic Service,8678,5011,0.577437,14.992308,39.536629,74,1,0,0.008527
3,Y56,R1H,BARTS HEALTH NHS TRUST,C_120,Ear Nose and Throat Service,6180,3002,0.485761,18.807339,46.870175,262,16,0,0.042395
4,Y56,R1H,BARTS HEALTH NHS TRUST,C_130,Ophthalmology Service,6249,4311,0.689870,11.591286,33.791613,4,0,0,0.000640
5,Y56,R1H,BARTS HEALTH NHS TRUST,C_140,Oral Surgery Service,3490,2092,0.599427,13.155963,41.103704,73,6,0,0.020917
6,Y56,R1H,BARTS HEALTH NHS TRUST,C_150,Neurosurgical Service,1467,757,0.516019,17.539216,43.868889,26,0,0,0.017723
7,Y56,R1H,BARTS HEALTH NHS TRUST,C_160,Plastic Surgery Service,1394,774,0.555237,14.612903,47.206667,60,5,0,0.043042
8,Y56,R1H,BARTS HEALTH NHS TRUST,C_170,Cardiothoracic Surgery Service,4,4,1.000000,-,-,0,0,0,0.000000
9,Y56,R1H,BARTS HEALTH NHS TRUST,C_300,General Internal Medicine Service,63,38,0.603175,14.125,31.98,0,0,0,0.000000


### Row grain

A row does **not** represent an individual patient.

Each row represents one:

**Provider + Treatment Function combination for July 2026**

The weekly columns contain aggregated counts of incomplete RTT pathways within waiting-time bands.


## Understand provider and treatment-function coverage


In [9]:
n_providers = df["Provider Code"].nunique()
n_treatment_functions = df["Treatment Function"].nunique()

print(f"Unique providers: {n_providers}")
print(f"Unique treatment functions: {n_treatment_functions}")
print(f"Providers × treatment functions: {n_providers * n_treatment_functions}")
print(f"Observed rows: {len(df)}")


Unique providers: 146
Unique treatment functions: 24
Providers × treatment functions: 3504
Observed rows: 3504


The dataset contains **146 providers** and **24 treatment-function categories**. Since `146 × 24 = 3,504`, every provider has one row for each of the 24 categories in this extract.


In [10]:
treatment_functions = (
    df[["Treatment Function Code", "Treatment Function"]]
    .drop_duplicates()
    .sort_values("Treatment Function Code")
    .reset_index(drop=True)
)

treatment_functions


,Treatment Function Code,Treatment Function
0,C_100,General Surgery Service
1,C_101,Urology Service
2,C_110,Trauma and Orthopaedic Service
3,C_120,Ear Nose and Throat Service
4,C_130,Ophthalmology Service
5,C_140,Oral Surgery Service
6,C_150,Neurosurgical Service
7,C_160,Plastic Surgery Service
8,C_170,Cardiothoracic Surgery Service
9,C_300,General Internal Medicine Service


## The `C_999` total row and double-counting risk

Each provider has a `C_999` row labelled **Total**. This row already aggregates the provider's treatment functions.

Therefore:

- use `C_999` when calculating **provider-level totals**;
- exclude `C_999` when analysing **individual specialties/treatment functions**; and
- never sum specialty rows together with the `C_999` row, because that would double-count activity.


In [11]:
df[df["Provider Code"] == "R1H"][
    [
        "Treatment Function Code",
        "Treatment Function",
        "Total number of incomplete pathways"
    ]
]


,Treatment Function Code,Treatment Function,Total number of incomplete pathways
0,C_100,General Surgery Service,4483
1,C_101,Urology Service,5901
2,C_110,Trauma and Orthopaedic Service,8678
3,C_120,Ear Nose and Throat Service,6180
4,C_130,Ophthalmology Service,6249
5,C_140,Oral Surgery Service,3490
6,C_150,Neurosurgical Service,1467
7,C_160,Plastic Surgery Service,1394
8,C_170,Cardiothoracic Surgery Service,4
9,C_300,General Internal Medicine Service,63


In [12]:
provider_totals = df[
    df["Treatment Function Code"] == "C_999"
].copy()

specialty_data = df[
    df["Treatment Function Code"] != "C_999"
].copy()

print("Provider total rows:", len(provider_totals))
print("Specialty/category rows:", len(specialty_data))


Provider total rows: 146
Specialty/category rows: 3358


## Basic data-quality checks

Before analysis, check:

- missing values;
- duplicate provider/treatment-function combinations; and
- data types.


In [13]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3504 entries, 0 to 3503
Columns: 119 entries, Region Code to % 52 plus weeks
dtypes: float64(2), int64(110), object(7)
memory usage: 3.2+ MB


In [14]:
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_values")
)

missing_summary.head(20)


,missing_values
Region Code,0
Provider Code,0
Provider Name,0
Treatment Function Code,0
Treatment Function,0
>0-1,0
>1-2,0
>2-3,0
>3-4,0
>4-5,0


In [15]:
duplicate_count = df.duplicated(
    subset=["Provider Code", "Treatment Function Code"]
).sum()

print("Duplicate provider/treatment-function combinations:", duplicate_count)


Duplicate provider/treatment-function combinations: 0


There are **no duplicate provider/treatment-function combinations** in the July 2026 Provider extract.


## Identify the weekly waiting-time bands

The columns from `>0-1` through `104 plus` represent weekly waiting-time bands.


In [16]:
weekly_columns = list(
    df.columns[
        df.columns.get_loc(">0-1"):
        df.columns.get_loc("104 plus") + 1
    ]
)

print("Number of weekly waiting-time bands:", len(weekly_columns))
print("First 10 bands:", weekly_columns[:10])
print("Last 10 bands:", weekly_columns[-10:])


Number of weekly waiting-time bands: 105
First 10 bands: ['>0-1', '>1-2', '>2-3', '>3-4', '>4-5', '>5-6', '>6-7', '>7-8', '>8-9', '>9-10']
Last 10 bands: ['>95-96', '>96-97', '>97-98', '>98-99', '>99-100', '>100-101', '>101-102', '>102-103', '>103-104', '104 plus']


There are **105 waiting-time bands**:

- `>0-1` means more than 0 weeks and up to 1 week;
- `>17-18` means more than 17 weeks and up to 18 weeks; and
- `104 plus` represents waits of 104 weeks or more.


## Reconcile the weekly bands to the published total

A strong data-understanding step is to independently sum all waiting-time bands and compare the result with NHS England's published `Total number of incomplete pathways`.


In [17]:
df["calculated_total"] = df[weekly_columns].sum(axis=1)

total_check = (
    df["calculated_total"]
    == df["Total number of incomplete pathways"]
)

print("Rows that reconcile:", total_check.sum())
print("Rows that do not reconcile:", (~total_check).sum())


Rows that reconcile: 3504
Rows that do not reconcile: 0


All **3,504 rows reconcile successfully**. This confirms that the weekly waiting-band counts sum exactly to the published incomplete-pathway total for every provider/treatment-function record.


## Validate the published 18-week measure

The first 18 weekly bands represent pathways waiting up to 18 weeks. We can independently calculate this figure and compare it with NHS England's `Total within 18 weeks`.


In [18]:
within_18_columns = weekly_columns[:18]

df["calculated_within_18"] = df[within_18_columns].sum(axis=1)

within_18_check = (
    df["calculated_within_18"]
    == df["Total within 18 weeks"]
)

print("Rows that reconcile:", within_18_check.sum())
print("Rows that do not reconcile:", (~within_18_check).sum())


Rows that reconcile: 3504
Rows that do not reconcile: 0


Again, all **3,504 rows reconcile successfully**. This provides a second independent validation of the published KPI.


## Identify non-numeric symbols in waiting-time statistics

The median and 92nd-percentile waiting-time columns are loaded as `object` rather than numeric values. Inspection shows that some records contain `-`.

We will not replace these values yet; handling them belongs in the data-cleaning notebook.


In [19]:
stats_columns = [
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)"
]

for col in stats_columns:
    hyphen_count = df[col].astype(str).str.strip().eq("-").sum()
    print(f"{col}")
    print(f"  dtype: {df[col].dtype}")
    print(f"  '-' values: {hyphen_count}")


Average (median) waiting time (in weeks)
  dtype: object
  '-' values: 1113
92nd percentile waiting time (in weeks)
  dtype: object
  '-' values: 1113


Both waiting-time statistic fields contain **1,113 `-` values**. This explains why pandas reads them as text/object columns and is an issue to address explicitly in `02_data_cleaning.ipynb`.


## Gloucestershire Hospitals NHS Foundation Trust case study

The dataset contains **Gloucestershire Hospitals NHS Foundation Trust** under provider code `RTE`. We will eventually use this provider as a local drill-down case study alongside national comparisons.


In [20]:
gloucestershire_total = df[
    (df["Provider Code"] == "RTE") &
    (df["Treatment Function Code"] == "C_999")
][
    [
        "Provider Code",
        "Provider Name",
        "Treatment Function",
        "Total number of incomplete pathways",
        "Total within 18 weeks",
        "% within 18 weeks",
        "Average (median) waiting time (in weeks)",
        "92nd percentile waiting time (in weeks)",
        "Total 52 plus weeks",
        "Total 65 plus weeks",
        "Total 78 plus weeks",
        "% 52 plus weeks"
    ]
]

gloucestershire_total


,Provider Code,Provider Name,Treatment Function,Total number of incomplete pathways,Total within 18 weeks,% within 18 weeks,Average (median) waiting time (in weeks),92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 65 plus weeks,Total 78 plus weeks,% 52 plus weeks
791,RTE,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,Total,68792,49568,0.720549,10.787426,33.831254,24,0,0,0.000349


In [21]:
gloucestershire_total.assign(
    **{
        "% within 18 weeks": gloucestershire_total["% within 18 weeks"] * 100,
        "% 52 plus weeks": gloucestershire_total["% 52 plus weeks"] * 100
    }
).round(2)


,Provider Code,Provider Name,Treatment Function,Total number of incomplete pathways,Total within 18 weeks,% within 18 weeks,Average (median) waiting time (in weeks),92nd percentile waiting time (in weeks),Total 52 plus weeks,Total 65 plus weeks,Total 78 plus weeks,% 52 plus weeks
791,RTE,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,Total,68792,49568,72.05,10.787426,33.831254,24,0,0,0.03


For July 2026, the provider total row reports:

- **68,792** incomplete pathways;
- **49,568** within 18 weeks;
- approximately **72.05%** within 18 weeks;
- median wait of approximately **10.79 weeks**;
- 92nd-percentile wait of approximately **33.83 weeks**;
- **24** pathways waiting 52+ weeks;
- **0** waiting 65+ weeks; and
- **0** waiting 78+ weeks.

At this stage these figures are descriptive only. Interpretation, benchmarking and trend analysis will come later.


## Data-understanding summary

### Confirmed structure

- Workbook contains six sheets.
- `Provider` is the primary sheet for this project.
- The real table header begins on Excel row 14.
- After removing one fully empty column, the sheet has **3,504 rows × 119 columns**.
- There are **146 providers**.
- There are **24 treatment-function categories per provider**.
- `C_999` is the provider-level total row.
- There are **105 weekly waiting-time bands** from `>0-1` to `104 plus`.
- Provider/treatment-function combinations are unique.
- Weekly band totals reconcile with the published total on all 3,504 rows.
- The first 18 weekly bands reconcile with the published 18-week total on all 3,504 rows.
- Median and 92nd-percentile waiting-time columns contain `-` values and require cleaning.

### Decisions carried forward to cleaning

In `02_data_cleaning.ipynb`, we will:

1. standardise column names;
2. convert `-` to appropriate missing values;
3. convert numeric columns to correct numeric data types;
4. add a reporting-period field for July 2026;
5. retain the distinction between provider totals and treatment-function rows;
6. create analyst-friendly KPI names;
7. repeat validation after cleaning; and
8. export a processed dataset for SQL and Power BI.
